# Exercise Sample Solution

The first step is to instrument the code. We can do this using `cProfile` to get a high level idea of which functions we should spend our time on first. Before editing the code, the `cProfile` output directs up to look at `get_prime_numbers` first as this takes up most of the runtime.

In [ ]:
# Instrumented version
import cProfile

def get_prime_numbers(n):
  #This function returns all prime numbers up to and including the value n
  primes=[]
  
  for candidate in range(2, n+1):
    #Initially assume each number is prime
    prime = True
    for factor in range(2, candidate):
      # Loop over every other number below candidate to see if the candidate is divisible by it
      if candidate % factor == 0:
        #If candidate is divisible by it then it is not prime
        prime = False

    if prime:
      # If candidate is prime, add it to the list of primes
      primes.append(candidate)

  return primes

def get_n_prime_factors(value):
  # This function calculates the number of prime factors of value
  #Get a list of primes whose values are less than value
  primes = get_prime_numbers(value)

  n_factors = 0

  for prime in primes:
    if value % prime == 0:
      #If the value is divisible by the prime then the prime is a prime factor of value
      n_factors = n_factors +1

  return n_factors

def fraction_power_sum(value):
  # This function returns the value of 1 + 1/value + 1/value**2 + 1/value**3 + ... + 1/value**100
  result=0

  for i in range(101):
    result = result + 1 / value**i

  return result

def main_calculation(n_max):
  # This is the main calculation
  # For each number up to n_max we check if it has at least 3 unique prime factors. If it does, we calculate the value of fraction_power_sum for this number and add the result to a running total

  result = 0

  for value in range(n_max + 1):
    if get_n_prime_factors(value) >= 3:
      result = result + fraction_power_sum(value)

  return result

#Call the function
cProfile.run('print(main_calculation(1000))')

The code cell below contains a sample solution with a number of optimisations. These include:

* Algorithm optimization in get_prime_numbers: Changed from checking all numbers from 2 to candidate as potential factors to only checking previously found primes as factors.
* Early loop termination in get_prime_numbers: Added a `break` statement when a factor is found, avoiding unnecessary checks.
* Eliminated redundant prime generation: Moved prime number calculation from `get_n_prime_factors` to main_calculation so primes are only calculated once instead of for every value
* Mathematical optimization in `fraction_power_sum`: Replaced the loop calculating a geometric series with the closed-form geometric sum formula: `(1 - 1/value**100) / (1 - 1/value)`
* Early loop termination in get_n_prime_factors: Added a `break` when the prime exceeds the value being checked, since larger primes cannot be factors

It may be possible to optimise further - can you beat the performance of this version?

In [ ]:
#@title
# A sample optimised version
import cProfile
from functools import lru_cache

def get_prime_numbers(n):
  primes=[]
  
  for candidate in range(2, n+1):
    prime = True

    #We only need to check to see if each prime is a factor of the candidate because if no primes are a factor, no other numbers will be
    for factor in primes:
      if candidate % factor == 0:
        prime = False
        #If candidate is divisible by it then it is not prime. We don't need to check if it has any other factors so we can break the loop
        break

    if prime:
      primes.append(candidate)

  return primes

def get_n_prime_factors(value, primes):
  n_factors = 0

  for prime in primes:
    # We now have a list of all primes under 1000 instead of all primes under value
    # As a result, we need to break the loop if prime is greater than value
    if prime > value:
      break
    if value % prime == 0:
      n_factors = n_factors +1

  return n_factors

def fraction_power_sum(value):
  #We can replace the for loop with the geometric sum equation
  return (1 - 1 / value ** 100) / (1 - 1 / value)

def main_calculation(n_max):
  result = 0

  #Calculate the primes here, rather than in get_n_prime_factors so they only need to be calculated once
  primes = get_prime_numbers(n_max + 1)

  for value in range(n_max + 1):
    # Now we pass primes to get_n_prime_factors
    if get_n_prime_factors(value, primes) >= 3:
      result = result + fraction_power_sum(value)

  return result

#Call the function
cProfile.run('print(main_calculation(1000))')